In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
from src.config import *

In [2]:
# 1. CARGA Y PRE-LIMPIEZA
# ==========================================
df_raw = pd.read_excel("../data/pacientes.xlsx")
hosp_coords = pd.read_csv("../data/hospitales_coordenadas.csv")
dict_comp = dict(zip(hosp_coords['Nombre Hospital'], hosp_coords['complejidad']))

df = df_raw.rename(columns={
    'Id': 'paciente_id', 'Nombre Hospital': 'hospital_origen',
    'Motivo': 'motivo_egreso', 'Fecha inicio': 'fecha_ingreso',
    'Fecha egreso': 'fecha_egreso', 'Edad': 'edad'
}).copy()

df['fecha_ingreso'] = pd.to_datetime(df['fecha_ingreso'], errors='coerce')
df['fecha_egreso'] = pd.to_datetime(df['fecha_egreso'], errors='coerce')

# ---> LA SOLUCIÓN: Convertir edad a número obligatoriamente <---
df['edad'] = pd.to_numeric(df['edad'], errors='coerce')

df = df.sort_values(['paciente_id', 'fecha_ingreso'])

In [3]:
# 2. FILTROS DE CALIDAD (Data Quality)
# ==========================================

# --- CRITERIO A: Consistencia de Edad ---
# Calculamos la diferencia, pero si un paciente tiene la edad en NaN, asumimos diferencia = 0
diff_edad = df.groupby('paciente_id')['edad'].agg(lambda x: x.max() - x.min() if x.notna().any() else 0)

# Filtramos (fillna(0) por si quedó algún nulo colgado)
pacientes_edad_ok = diff_edad[diff_edad.fillna(0) <= TOLERANCIA_EDAD_ANIOS].index
print(f"Pacientes descartados por inconsistencia de edad: {len(diff_edad) - len(pacientes_edad_ok)}")

# --- CRITERIO B: Desenlace Válido ---
def obtener_fin_caso(g):
    # Si motivo_egreso es nulo, le ponemos un string vacío para que no rompa el .str.lower()
    motivos = g['motivo_egreso'].fillna('').str.lower()
    g['prioridad'] = motivos.map(ORDEN_PRIORIDAD_CLINICA).fillna(99)
    return g.sort_values(['prioridad', 'fecha_egreso'], ascending=[True, False]).iloc[0]['motivo_egreso']

desenlaces = df.groupby('paciente_id').apply(obtener_fin_caso).reset_index(name='motivo_final')

pacientes_final_ok = desenlaces[~desenlaces['motivo_final'].isin(MOTIVOS_A_DESCARTAR)].paciente_id
print(f"Pacientes descartados por final inconcluso: {len(desenlaces) - len(pacientes_final_ok)}")

# EL FILTRO MAESTRO
lista_pacientes_validos = set(pacientes_edad_ok).intersection(set(pacientes_final_ok))

Pacientes descartados por inconsistencia de edad: 52
Pacientes descartados por final inconcluso: 4093


In [4]:
# 3. GENERACIÓN DE LOS 3 DATAFRAMES
# ==========================================

# BASE 1: df_base_limpia (Episodios curados)
df_base_limpia = df[df['paciente_id'].isin(lista_pacientes_validos)].copy()

# BASE 2: df_traslados (Solo los saltos)
df_base_limpia['hosp_destino'] = df_base_limpia.groupby('paciente_id')['hospital_origen'].shift(-1)
df_base_limpia['fecha_ingreso_destino'] = df_base_limpia.groupby('paciente_id')['fecha_ingreso'].shift(-1)
df_base_limpia['gap_dias'] = (df_base_limpia['fecha_ingreso_destino'] - df_base_limpia['fecha_egreso']).dt.days

mask_traslado = (
    df_base_limpia['hosp_destino'].notna() & 
    (df_base_limpia['hospital_origen'] != df_base_limpia['hosp_destino']) & 
    (df_base_limpia['gap_dias'] <= DIAS_VENTANA_TRASLADO)
)
df_traslados = df_base_limpia[mask_traslado].copy()

# BASE 3: df_pacientes_trayectorias (Un paciente por fila)
df_base_limpia['complejidad'] = df_base_limpia['hospital_origen'].map(dict_comp).astype(str)

df_pacientes_trayectorias = df_base_limpia.groupby('paciente_id').agg(
    recorrido_hospitales=('hospital_origen', lambda x: ' -> '.join(x)),
    recorrido_complejidad=('complejidad', lambda x: ' -> '.join(x)),
    cantidad_hospitales=('hospital_id', 'count'),
    dias_totales_red=('dias_en_nodo', 'sum'),
    fecha_inicio_red=('fecha_ingreso', 'first'),
    fecha_fin_red=('fecha_egreso', 'last')
).reset_index()

# Pegamos el motivo final estandarizado
df_pacientes_trayectorias = df_pacientes_trayectorias.merge(desenlaces, on='paciente_id')
df_pacientes_trayectorias['desenlace'] = df_pacientes_trayectorias['motivo_final'].map(MAPEO_NOMBRES_FINALES)

KeyError: "Label(s) ['dias_en_nodo', 'hospital_id'] do not exist"

In [ ]:
# ==========================================
# 4. EXPORTACIÓN A EXCEL
# ==========================================
with pd.ExcelWriter("../results/Base_Procesada_Red.xlsx") as writer:
    df_pacientes_trayectorias.to_sheet(writer, sheet_name='Trayectorias_Pacientes', index=False)
    df_base_limpia.to_sheet(writer, sheet_name='Base_Episodios_Limpia', index=False)
    df_traslados.to_sheet(writer, sheet_name='Solo_Traslados', index=False)

print("\n¡Pipeline completado! Archivo generado en /results/Base_Procesada_Red.xlsx")